# SUB-009 и SUB-010 — stacking и MLP-024

Notebook воспроизводит два Kaggle-файла:

- `SUB-009`: ENS-003 с автоматически обученными весами;
- `SUB-010`: принятый чемпион MLP-024.

Для stacking финальные веса учатся на cross-fitted OOF-вероятностях train. Затем базовые модели обучаются на полном train и предсказывают test. MLP обучается один раз и используется в обоих submission.


## 1. Окружение

Полный прогон перезаписывает SUB-009 и SUB-010 тем же воспроизводимым контрактом.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / 'README.md').exists() and (candidate / 'src').exists()
)
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.ensemble_submission import create_ensemble_and_mlp_submissions
print(PROJECT_ROOT)


## 2. Full-train fit и сохранение

Ячейка обучает XGBoost, LightGBM, CatBoost и MLP-024. Ограничьте BLAS/OMP одним потоком перед запуском Jupyter, если окружение снова начинает расходовать много памяти.


In [ ]:
stacking_submission, mlp_submission = create_ensemble_and_mlp_submissions(
    PROJECT_ROOT,
    stacking_id='SUB-009',
    mlp_id='SUB-010',
)
print(stacking_submission)
print(mlp_submission)


## 3. Проверка файлов

Оба файла должны иметь 418 строк, два столбца, уникальный `PassengerId` и только классы `0/1`.


In [ ]:
stacking = pd.read_csv(stacking_submission.csv_path)
mlp = pd.read_csv(mlp_submission.csv_path)

checks = pd.DataFrame([
    {'submission': 'SUB-009', 'rows': len(stacking), 'unique_keys': stacking.PassengerId.nunique(), 'missing': stacking.Survived.isna().sum(), 'survived_1': int(stacking.Survived.sum())},
    {'submission': 'SUB-010', 'rows': len(mlp), 'unique_keys': mlp.PassengerId.nunique(), 'missing': mlp.Survived.isna().sum(), 'survived_1': int(mlp.Survived.sum())},
])
display(checks)
print('Различающихся ответов:', int((stacking.Survived != mlp.Survived).sum()))
display(stacking.head())
display(mlp.head())


## 4. Финальные веса stacking

Эти веса обучены для финального test-serving на всех доступных OOF-прогнозах train. Они не являются результатом внешней оценки ENS-003: честная CV-метрика по-прежнему берётся из nested CV.


In [ ]:
weights = pd.read_csv(PROJECT_ROOT / 'artifacts/submissions/SUB-009/final_weights.csv')
display(weights.round(4))
